In [1]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import os


Define parameters


In [4]:
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io
import sys
import os


def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    
    df.index = df.index.tz_convert(local_tz)
    return df

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)

    return df.loc[(df.index >= start_date) & (df.index <= end_date)]

def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    header = '\n'.join(response.text.splitlines()[:8])

    # Check if the dataframe has more than 8761 rows and truncate if necessary
    # Sometimes MERRA2 erroneously provides extra rows
    if len(df) > 8760:
        df = df.iloc[:8760]


    return df, header

# def merge_data(df, data):
#     """
#     Merges two datasets, replacing specific columns in `df` with corresponding values from `data`.
#     """

#     print(df.head(5))
#     for col in [6, 7, 8, 33, 30, 21, 20, 9]:  # Replace indices with more descriptive names if possible
#         if not df[col].isna().all():
#             df[col] = list(data['temp'][1:])
#     print(df.head(5))
#     return df


def merge_data(df, data):
    # Tdb
    if not data['temp'].isna().all():
        df[6] = list(data['temp'][1:])
    # Tdew
    if not data['dwpt'].isna().all():
        df[7] = list(data['dwpt'][1:])
    # RH
    if not data['rhum'].isna().all():
        df[8] = list(data['rhum'][1:])
    # Precep
    if not data['prcp'].isna().all():
        df[33] = list(data['prcp'][1:])
    # Snow
    if not data['snow'].isna().all():
        df[30] = list(data['snow'][1:])
    # Wspeed
    if not data['wspd'].isna().all():
        df[21] = list(data['wspd'][1:])
    # Wdir
    if not data['wdir'].isna().all():
        df[20] = list(data['wdir'][1:])
    # P, go from hPa to Pa
    if not data['pres'].isna().all():
        df[9] = [x * 100 for x in list(data['pres'][1:])]

    return df

def check_missing_hours(year, df):
    """
    Checks for missing hours in the DataFrame's datetime index for a specified year.
    """
    full_index = pd.date_range(start=f"{year}-01-01", end=f"{year+1}-01-01", freq="H")
    missing_hours = full_index.difference(df.index)
    missing_hours_num = len(missing_hours)

    if missing_hours_num > 0:
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        largest_consecutive_group = (diffs != 1).cumsum().value_counts().max()
    else:
        largest_consecutive_group = 0

    return missing_hours_num, largest_consecutive_group

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='forward')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)

    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

def run_individual_location(lat, lon, year, file_type, save_folder, save_name):
    """
    Processes a single location, fetching data and handling errors.
    """
    data_meteostat_merra2, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)
    if epw_exists:
        retrieve_status = False
        distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        retrieve_info_closest_other_locations = True
        # return retrieve_status, distance, info_dict['wmo'], hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations
    elif retrieve_status:
        retrieve_info_closest_other_locations = False
        #Save the EPW file
        if save_name != None:
            output_path = os.path.join(save_folder, f"{save_name.replace(' ', '_').replace('.', '_')}_{year}.epw")
        else:
            output_path = os.path.join(save_folder, f"{wmo}_{year}.epw")
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)
        with open(output_path, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_path, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
    else:
        retrieve_info_closest_other_locations = False
        retrieve_status = False
        print('No data available for this location/year.')

    return retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations

def get_time_shift(timezone_name):
    """
    Calculates the time shift for a given timezone from UTC.
    """
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    utc_offset = now.utcoffset()
    return int(utc_offset.total_seconds() // 3600)  # Return hours offset only

def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.
    """
    df[temperature_column + '_F'] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65

    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column + '_F'].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']

    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)

def check_epw_exists(save_folder, year, wmo):
    return os.path.exists(f'{save_folder}/{wmo}_{year}.epw')

def create_header(df, year, info_dict):
    header_lines = []

    #Calculated parameters 
    first_day_year = pd.to_datetime(date.min.replace(year=year)).day_name()
    leap_status = lambda year: 'Yes' if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 'No'
    dst_start, dst_end = get_dst_start_end(year, info_dict['lat'], info_dict['lon'])
    design_conditions_file = 'resources/design_conditions.csv'
    design_conditions_line = find_closest_design_condition(float(info_dict['lat']), float(info_dict['lon']), design_conditions_file)
    #Hardcoded parameters
    number_of_holidays = 0
    number_of_data_periods = 1
    number_of_records_per_hour = 1

    # line_1
    header_lines.append(f"LOCATION,{info_dict['station_name']},{info_dict['state']},{info_dict['country']},{info_dict['weather_file_type']},{info_dict['wmo']},{info_dict['lat']},{info_dict['lon']},{info_dict['timeshift']},{info_dict['elevation']}")
    # line_2
    header_lines.append(design_conditions_line)
    # line_3
    header_lines.append(f"TYPICAL/EXTREME PERIODS,0")
    # line_4
    header_lines.append(f"GROUND TEMPERATURES,0")
    # header_lines.append(f"GROUND TEMPERATURES,3,.5,,,,-16.34,-17.80,-15.22,-11.16,-0.57,7.61,13.13,14.81,11.95,5.60,-2.89,-10.76,2,,,,-10.97,-13.57,-13.04,-10.89,-3.80,2.61,7.74,10.49,9.90,6.30,0.46,-5.74,4,,,,-6.53,-9.19,-9.78,-8.97,-4.96,-0.64,3.32,6.08,6.72,5.16,1.73,-2.4")
    # line_5
    try:
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},{dst_start.month}/{dst_start.day},{dst_end.month}/{dst_end.day},{number_of_holidays}")
    except AttributeError:
        #We cannot retrieve DST dates, let's set them to 0
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},0,0,{number_of_holidays}")
    # line_6
    header_lines.append(f"COMMENTS 1, ")
    # line_7
    header_lines.append(f"COMMENTS 2, ")
    # line_8
    header_lines.append(f"DATA PERIODS,{number_of_data_periods},{number_of_records_per_hour},Data,{first_day_year},{df.iloc[0, 1]}/{df.iloc[0, 2]},{df.iloc[-1, 1]}/{df.iloc[-1, 2]}")

    return header_lines

def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    try:
        return str(int(wmo))
    except ValueError:
        icao = wmo
        icao_converted = get_wmo_from_icao_NOAA(icao)
        if isinstance(icao_converted, type(None)):
            return icao
        else:
            return icao_converted

    # return wmo

def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        if (len(data.index) >100) & (station_number>1):
            data = data.loc[data.index.get_level_values('station').unique()[-1]]


        # Fetching hourly data for those stations
        # data = Hourly(stations.fetch(2), start, end, model=True).fetch()
        # Hourly(stations.fetch(2), start, end, model=True).fetch().loc[Hourly(stations.fetch(2), start, end, model=True).fetch().index.get_level_values('station').unique()[-1]]



        # print('+++++++')
        # print(lat)
        # print(lon)
        # print(start)
        # print(end)
        
        len_data = len(data.index)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        distance = stations.fetch(station_number)['distance'].values[-1]
        # print('distance')
        # print(distance)
        # print('len_data')
        # print(len(data.index))
        # print(largest_consecutive_group)
        # print(incomplete_timeseries)
        # Let's stop after 100mi
        if distance > 16093400:
            break
        
    try:
        if distance is not None:
            print('Distance')
            print(distance)
    except UnboundLocalError:
        print("Distance variable is not defined yet.")


    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries

# Helper function to check and update missing data
def update_if_missing(df, index, col_name, new_value):
    if pd.isna(df.at[index, col_name]) or not df.at[index, col_name]:
        df.at[index, col_name] = new_value

def retrieve_info_other_location(wmo, zipcodes, year):
    retrieve_status = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"Do we have data for {year}?"].values[0]
    distance = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"distance_location_station_miles_{year}"].values[0]
    hdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"hdd_base65F_{year}"].values[0]
    cdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"cdd_base65F_{year}"].values[0]
    return retrieve_status, distance, hdd, cdd

def get_wmo_from_icao_NOAA(icao_code):
    # Path to the local CSV file in the resource folder
    csv_file_path = os.path.join(os.path.join(os.getcwd(), 'resources'), 'isd-history.csv')

    # Read the CSV file
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            headers = lines[0].split(',')
            icao_index = headers.index('"ICAO"')
            wmo_index = headers.index('"USAF"')

            for line in lines[1:]:
                fields = line.split(',')
                if fields[icao_index].strip('"') == icao_code.upper():
                    return fields[wmo_index].strip('"')

    except FileNotFoundError:
        print(f"CSV file not found at path: {csv_file_path}")
        return None
    except Exception as e:
        # print(f"An error occurred: {e}")
        return None

def get_dst_start_end(year, latitude, longitude):
    # Get the timezone for the given latitude and longitude
    tf = TimezoneFinder()
    timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    
    if timezone_str is None:
        raise ValueError("Could not find timezone for the given coordinates.")
    
    # Get the timezone object
    timezone = pytz.timezone(timezone_str)
    
    # Define the dates for the beginning and end of the year (naive datetime)
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year, 12, 31)
    
    dst_start = None
    dst_end = None

    # Start by localizing the first date
    previous_offset = timezone.localize(start_of_year).dst()

    # Loop through each day of the year
    for dt in [start_of_year + timedelta(days=i) for i in range((end_of_year - start_of_year).days + 1)]:
        localized_dt = timezone.localize(dt)  # Localize naive datetime
        current_offset = localized_dt.dst()
        
        if previous_offset == timedelta(0) and current_offset != timedelta(0):
            dst_start = localized_dt
        elif previous_offset != timedelta(0) and current_offset == timedelta(0):
            dst_end = localized_dt
            break
        
        previous_offset = current_offset
    
    return dst_start, dst_end

def find_closest_design_condition(lat,lon,design_conditions_file):
    """
    Finds the closest design condition from the CSV file based on the given latitude and longitude.

    Parameters:
    lat (float): The latitude of the target location.
    lon (float): The longitude of the target location.
    csv_file (str): The path to the CSV file containing design conditions.

    Returns:
    str: The 2021 design condition string for the closest location.
    """

    # Function to calculate the distance between two points given their latitudes and longitudes
    def haversine_distance(lat1, lon1, lat2, lon2):
        # Convert latitude and longitude from degrees to radians
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        
        # Haversine formula
        dlat = lat2 - lat1 
        dlon = lon2 - lon1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        r = 6371  # Radius of Earth in kilometers. Use 3956 for miles. Determines return value units.
        return c * r

    
    # Read the CSV file
    df = pd.read_csv(design_conditions_file)

    # Calculate distance from target coordinates to each row in the dataframe
    df['distance'] = df.apply(lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']), axis=1)

    # Find the row with the minimum distance
    closest_row = df.loc[df['distance'].idxmin()]

    # Return the design conditions for 2021
    return closest_row['2021_design_conditions']


# Define constants
year = 2022
file_type = 'AMY'
save_folder = 'epws_wmo'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[8472:].iterrows():
    # if float(row.get(f"distance_location_station_miles_{year}")) < 50:
    #     continue
    print(index)


    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            print(wmo)
            retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)


    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"Do we have data for {year}?", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance * 0.000621371)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv('resources/zip_code_list.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

# KVHN0
# KSUN
# 74611


8472
20899
Distance
30343.45220590613
8473
54666
Distance variable is not defined yet.
8474
51011
Distance variable is not defined yet.
8475
15532
Distance variable is not defined yet.
8476
95464
Distance variable is not defined yet.
8477
17403
Distance
13950.417188101213
8478
50104
Distance variable is not defined yet.
8479
35070
Distance variable is not defined yet.
8480
50619
Distance variable is not defined yet.
8481
40063
Distance
11311.96085368961
8482
47114
Distance variable is not defined yet.
8483
57024
Distance variable is not defined yet.
8484
20879
Distance
30321.557209302187
8485
16680
Distance variable is not defined yet.
8486
49340
Distance variable is not defined yet.
8487
81225
Distance variable is not defined yet.
8488
46943
Distance
30309.288502393687
8489
47457
Distance variable is not defined yet.
8490
74001
Distance
28056.799871180538
8491
95364
Distance variable is not defined yet.
8492
29453
Distance variable is not defined yet.
8493
96094
Distance
28969.9296743

Distance
21910.169754817434
8802
35063
Distance variable is not defined yet.
8803
32693
Distance variable is not defined yet.
8804
35579
Distance variable is not defined yet.
8805
31744
Distance
29763.777336007086
8806
72129
Distance variable is not defined yet.
8807
10596
Distance
29560.57747628495
8808
72903
Distance
25163.22316201514
8809
26412
Distance
22897.02160715369
8810
71653
Distance variable is not defined yet.
8811
48472
Distance variable is not defined yet.
8812
60931
Distance variable is not defined yet.
8813
46996
Distance
29754.77051807348
8814
61321
Distance variable is not defined yet.
8815
62860
Distance
29753.457025243668
8816
63441
Distance variable is not defined yet.
8817
57646
Distance variable is not defined yet.
8818
65035
Distance variable is not defined yet.
8819
30568
Distance variable is not defined yet.
8820
61553
Distance variable is not defined yet.
8821
29569
Distance
29738.22097264095
8822
78107
Distance variable is not defined yet.
8823
24836
Distanc

Distance
26391.80483661688
8902
25514
Distance variable is not defined yet.
8903
95242
Distance variable is not defined yet.
8904
67071
Distance variable is not defined yet.
8905
59833
Distance variable is not defined yet.
8906
63943
Distance variable is not defined yet.
8907
41531
Distance
27857.436915536146
8908
30576
Distance variable is not defined yet.
8909
54466
Distance variable is not defined yet.
8910
94576
Distance variable is not defined yet.
8911
98572
Distance variable is not defined yet.
8912
95922
Distance variable is not defined yet.
8913
17352
Distance variable is not defined yet.
8914
77417
Distance variable is not defined yet.
8915
60468
Distance variable is not defined yet.
8916
62615
Distance variable is not defined yet.
8917
35184
Distance
26021.047872227227
8918
36264
Distance variable is not defined yet.
8919
24941
Distance variable is not defined yet.
8920
68332
Distance variable is not defined yet.
8921
98253
Distance
19821.536198786544
8922
17362
Distance
753

Distance
19832.8788897715
9041
04271
Distance variable is not defined yet.
9042
24574
Distance variable is not defined yet.
9043
97031
Distance
26559.028229471678
9044
45647
Distance variable is not defined yet.
9045
76652
Distance variable is not defined yet.
9046
82432
Distance variable is not defined yet.
9047
73662
Distance variable is not defined yet.
9048
16946
Distance variable is not defined yet.
9049
16852
Distance variable is not defined yet.
9050
59727
Distance variable is not defined yet.
9051
78619
Distance variable is not defined yet.
9052
78385
Distance variable is not defined yet.
9053
40046
Distance variable is not defined yet.
9054
56347
Distance
29400.940640471625
9055
96015
Distance variable is not defined yet.
9056
43756
Distance variable is not defined yet.
9057
54651
Distance
29393.864263052637
9058
68874
Distance
26772.784982055666
9059
05775
Distance variable is not defined yet.
9060
82941
Distance variable is not defined yet.
9061
64643
Distance variable is no

Distance
29368.217852858186
9081
85603
Distance variable is not defined yet.
9082
61951
Distance variable is not defined yet.
9083
74340
Distance variable is not defined yet.
9084
40350
Distance
27149.956621050653
9085
85540
Distance variable is not defined yet.
9086
17035
Distance variable is not defined yet.
9087
74467
Distance variable is not defined yet.
9088
12428
Distance variable is not defined yet.
9089
49079
Distance variable is not defined yet.
9090
63056
Distance variable is not defined yet.
9091
04673
Distance variable is not defined yet.
9092
67120
Distance
15655.376986433837
9093
53932
Distance variable is not defined yet.
9094
62931
Distance
26481.949511749295
9095
15135
Distance
10783.190754204938
9096
10952
Distance
29344.89919157575
9097
54748
Distance variable is not defined yet.
9098
47142
Distance variable is not defined yet.
9099
95363
Distance
11314.292975642462
9100
54810
Distance
29335.70529783726
9101
36523
Distance variable is not defined yet.
9102
71667
Dist

...

Distance
13266.10653144342
32959
99566


Distance
134106.98456298513
32960
99586
Distance
89814.91619005115
32961
99627


Distance
80890.83147273831
32962
99638


Distance
8079.798123994319
32963
99691


Distance
96956.74319018351
32964
99701
Distance
59581.889323164454
32965
99742
Distance
2735.039502208858


In [5]:

def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    try:
        return str(int(wmo))
    except ValueError:
        icao = wmo
        icao_converted = get_wmo_from_icao_NOAA(icao)
        if isinstance(icao_converted, type(None)):
            return icao
        else:
            return icao_converted

    # return wmo

fix_wmo('PAHV0')

'PAHV0'

In [8]:
print(zipcodes.columns)

Index(['zip', 'lat', 'lng', 'city', 'state_id', 'state_name', 'zcta',
       'parent_zcta', 'population', 'density', 'county_fips', 'county_name',
       'county_weights', 'county_names_all', 'county_fips_all', 'imprecise',
       'military', 'timezone', 'Distance [km]', 'Distamce [mi]', 'Location',
       'zip0', 'Do we have data for 2022?', 'weather_station_wmo_2022',
       'hdd_base65F_2022', 'cdd_base65F_2022',
       'distance_location_station_miles_2022'],
      dtype='object')


In [5]:
meteostat_df = pd.read_csv('resources/meteostat_stats.csv')
zipcodes = pd.read_csv('resources/zip_code_list.csv')
meteostat_df


,id,name,country,region,wmo,icao,latitude,longitude,elevation,timezone,hourly_start,hourly_end,daily_start,daily_end,monthly_start,monthly_end,distance
0,ZEZ8W,Pinhorn AGCM,CA,AB,NaN,NaN,49.0300,-111.0300,1061.0,America/Denver,2020-01-01,2022-12-14,2018-05-13,2022-12-11,2018-01-01,2022-01-01,4.182579e+04
1,71345,Masinasin Agdm,CA,AB,71345.0,CXMN,49.1400,-111.6500,948.0,America/Edmonton,2004-08-24,2024-05-09,2002-04-01,2024-04-28,2002-01-01,2022-01-01,5.058997e+04
2,71070,Pakowki Lake AGCM,CA,AB,71070.0,CPPL,49.2200,-111.1300,915.0,America/Edmonton,2008-04-01,2024-05-28,2007-09-01,2024-04-28,2007-01-01,2022-01-01,5.763066e+04
3,71244,Milk River,CA,AB,71244.0,CWRY,49.1300,-112.0500,1050.0,America/Edmonton,1993-09-08,2024-09-22,1993-09-09,2024-04-28,1994-01-01,2022-01-01,6.728820e+04
4,72769,Cut Bank / Little Browning,US,MT,72769.0,KCTB,48.6084,-112.3761,1176.0,America/Denver,2005-01-01,2024-09-22,1903-12-01,2024-12-30,1903-01-01,2022-01-01,7.535992e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15786,61996,Martin De Vivies Ile Amsterdam,TF,NaN,61996.0,NaN,-37.8000,77.5333,27.0,Europe/Paris,NaN,NaN,1973-01-15,2019-12-27,1954-01-01,2021-01-01,1.860485e+07
15787,61997,Alfred Faure Iles Crozet,TF,NaN,61997.0,NaN,-46.4333,51.8667,143.0,Europe/Paris,NaN,NaN,1976-06-06,2023-12-30,1975-01-01,2021-01-01,1.873456e+07
15788,94997,Heard Island The Spit,AU,NaN,94997.0,NaN,-53.1000,73.7167,12.0,Pacific/Norfolk,NaN,NaN,1986-12-02,2015-11-17,1999-01-01,2015-01-01,1.941244e+07
15789,95997,Atlas Cove (Heard Island),NF,NaN,95997.0,NaN,-53.0189,73.3917,3.0,Pacific/Norfolk,NaN,NaN,1998-03-01,2012-12-20,1999-01-01,2012-01-01,1.943282e+07


In [ ]:
zip_row.columns

In [6]:
import math

# Function to calculate the distance between two lat/lon points using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in miles
    R = 3958.8
    
    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in miles
    return R * c


# Iterate over each row in zip_df
for idx, zip_row in zipcodes.iterrows():
    # print(idx)

    if bool(zip_row['Do we have data for 2022?']):
        wmo_code = zip_row['weather_station_wmo_2022']

        # print(wmo_code)
        
        # Look for a matching row in meteostat_df using WMO or ICAO code
        # station_row = meteostat_df[(meteostat_df['wmo'].astype(str) == wmo_code) | (meteostat_df['icao'].astype(str) == wmo_code)]
        try:
            station_row = meteostat_df[(meteostat_df['id'].astype(str) == str(wmo_code))]
        except ValueError:
            try:
                station_row = meteostat_df[(meteostat_df['id'].astype(str) == wmo_code[:4])]
            except TypeError:
                print('----------------------------')
                print(wmo_code)
                print(zip_row['zip0'])
                print(zip_row['Do we have data for 2022?'])
                continue
                # station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
               
        



        if not station_row.empty:
            # Extract lat/lon from meteostat_df
            lat_stat = station_row['latitude'].values[0]
            lon_stat = station_row['longitude'].values[0]
            
            # Extract lat/lon from zip_df
            lat_zip = zip_row['lat']
            lon_zip = zip_row['lng']
            
            # Calculate the distance in miles
            distance = haversine(lat_zip, lon_zip, lat_stat, lon_stat)
            
            # Save the calculated distance in the new column
            zipcodes.at[idx, 'distance_location_station_miles_2022___'] = distance

# Show the updated zip_df with distances
zipcodes.head()


,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,...,Distance [km],Distamce [mi],Location,zip0,Do we have data for 2022?,weather_station_wmo_2022,hdd_base65F_2022,cdd_base65F_2022,distance_location_station_miles_2022,distance_location_station_miles_2022___
0,89017,37.73096,-115.24786,Hiko,NV,Nevada,True,NaN,102,0.1,...,65.984144,40.983940,Caliente,89017,True,74614,3249.0,2749.0,11739.216788,82.582356
1,99743,63.78380,-150.12799,Healy,AK,Alaska,True,NaN,1041,0.1,...,57.614148,35.785185,Healy River,99743,True,PAIN0,13417.0,4.0,11988.055687,37.359366
2,89001,37.27386,-115.39051,Alamo,NV,Nevada,True,NaN,1441,0.3,...,68.677573,42.656878,Nevada Test Site / Sugar Bunker,89001,True,74614,3249.0,2749.0,7.294409,50.092802
3,84631,38.95992,-112.38345,Fillmore,UT,Utah,True,NaN,3066,7.2,...,44.952789,27.920987,Delta,84631,True,72479,6114.0,1233.0,11894.009947,29.818768
4,57702,44.03379,-103.39048,Rapid City,SD,South Dakota,True,NaN,34801,46.0,...,26.554668,16.493582,Ellsworth Air Force Base,57702,True,KSPF0,7509.0,722.0,12003.397916,36.582680


In [8]:
meteostat_df[(meteostat_df['wmo'] == int('71345'))]

,name,country,region,wmo,icao,latitude,longitude,elevation,timezone,hourly_start,hourly_end,daily_start,daily_end,monthly_start,monthly_end,distance


In [7]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)


In [19]:


Stations().nearby(48.72526, -111.36528).fetch().to_csv('resources/meteostat_stats.csv', index=True)

In [ ]:
def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        distance = stations.fetch(station_number)['distance'].values[-1]
        # Let's stop after 100mi
        if distance > 160000:
            break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries


lat = 42.06259
lon = -72.62589

data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries = get_data_noaa(lat, lon, 2022, '')
data.head(5)

In [ ]:
data.interpolate(method='linear', limit=3, limit_direction='forward')

In [ ]:
zipcodes.head(20)

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data